# Library matches of the clustered spectra and their manual classification

Downloads the GNPS2 library-search results of every core (cosine ≥ 0.7, ≥ 6 matched peaks, 0.02 Da) and attaches the manual classification `Is_BA_Checked` (steroidal core = yes, other class = no). The per-core tables written at the end are the annotation input for the network filter in `04_entropy_MN_filters_and_deltas`; the combined table is also used for Supplementary Figure 5.

In [ ]:
import os

# Root of the local project folder (the one holding Falcon/, Queries_final_mgf/,
# final_mgfs_library_generation/, ...). Edit this single line to run elsewhere.
PROJECT_DIR = '/media/dorrestein/Helena2/Bile_acids_update'

In [ ]:
import pandas as pd
import requests
import os
from pathlib import Path
import time

In [ ]:
# All the GNPS classical molecular networking task IDs for each core

core_tasks = {
    "C24 - Nonhydroxy": "c5f057071d0d433db641b3588e4640b3",
    "C24 - Monohydroxy": "3d4b17d7cffd4912b4d5ac9e4d96704a",
    "C24 - Dihydroxy": "60b85ed0b0ed4ebe95086b2e7bffa019",
    "C24 - Trihydroxy (part 1)": "1d57b925cf134c6f8b55928f722e698b",
    "C24 - Trihydroxy (part 2)": "27e0b721885748ecb68eeb4579d9c3db",
    "C24 - Trihydroxy (part 3)": "4e7ea04ff6b144c6af46463c3e249935",
    "C24 - Tetrahydroxy": "bb84d48eb0ba4f4f80223a819f895d28",
    "C24 - Pentahydroxy": "cae1a6421b674f259fbaea4364c020c0",
    "C23 - Monohydroxy": "c09865fdb2e44613bd29fb4599b01877",
    "C23 - Dihydroxy": "12de067b1ab64aaf9b7cf0e9f51edd29",
    "C23 - Trihydroxy": "061d6f8efaf74187b55f4612b3649c06",
    "C23 - Tetrahydroxy": "f7a7f7320c664eef9a2a710b3e5171d2",
    "C23 - Pentahydroxy": "fe48e9a75c814a749115a1b721150c4e",
    "C27 - Monohydroxy": "6d2d424aac0b4786aec703445e9bba62",
    "C27 - Dihydroxy": "65c0038557b7411aadee6713f3c79d3e",
    "C27 - Trihydroxy": "43db7eb70e21467d9be588980eb2d799",
    "C27 - Tetrahydroxy": "42bbeaed8b564d0390efc1f256f66536",
    "C27 - Pentahydroxy": "e2109930c569483b97c8d101ccd5443d",
    "Aceto dihydroxy": "ecc7e816a2dd40ce9ebc47bf54049665",
    "C24 alkylamine - Monohydroxy": "c12df5f6e45942308a821155e3056909",
    "C24 alkylamine - Dihydroxy": "870020c71ad94ac3a1eb227af821c55f",
    "C24 alkylamine - Trihydroxy (part 1)": "6d12d01e0cab4223918e10fe5e341484",
    "C24 alkylamine - Trihydroxy (part 2)": "582018354469488e9957eb59e4ba1f7a",
    "C24 alkylamine - Trihydroxy (part 3)": "b5f05dc0f34445e1aab052d185f34dc7",
    "C24 alkylamine - Trihydroxy (part 4)": "821cda30537c4aafb7c6c3399a8a03e8",
    "C24 alkamine - Tetrahydroxy": "c1e5b49287df4e5596a1849ecfe33c3a",
    "C24 alkylamine - Pentahydroxy": "ed10f53be4f14ec9bcdfa35092213b3f",
    "C28 alcohol - 1OH": "d7fda9fc87d14c5eb8037f8b848bf455",
    "C28 alcohol - 2OH": "d06e207cbc124f949ff5c9d03715b33b",
    "C28 alcohol - 3OH": "e8bb4dda3a7b49e2b6e5b7df3b206638"
}

# folder where the individual GNPS2 result files are saved
temp_dir = Path(f'{PROJECT_DIR}/library_results_postMN_filter/library_matches_files')
temp_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Download all TSV files
print("Downloading library match files...")
all_dataframes = []

for core_name, task_id in core_tasks.items():
    url = f"https://gnps2.org/resultfile?task={task_id}&file=nf_output/library/merged_results_with_gnps.tsv"
    file_path = temp_dir / f"{task_id}.tsv"
    
    print(f"Downloading {core_name}...")
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        # Save to file
        with open(file_path, 'wb') as f:
            f.write(response.content)
        
        # Read the TSV file
        df = pd.read_csv(file_path, sep='\t')
        
        # Add the core column and URL
        status_url = f"https://gnps2.org/status?task={task_id}"
        df.insert(0, 'Core', core_name)
        df.insert(1, 'Status_URL', status_url)
        
        all_dataframes.append(df)
        print(f"  ✓ Downloaded {len(df)} rows")
        
        # Small delay to be respectful to the server
        time.sleep(0.5)
        
    except Exception as e:
        print(f"  ✗ Error downloading {core_name}: {str(e)}")

# Concatenate all dataframes
combined_df = pd.concat(all_dataframes, ignore_index=True)

In [ ]:
# Save the combined file
output_file = f"{PROJECT_DIR}/Falcon/postMN_final_filter/combined_library_matches.tsv"
combined_df.to_csv(output_file, sep='\t', index=False)
print(f"\nSaved combined data to: {output_file}")

In [ ]:
# add a classification column after manual inspection

combined_df = pd.read_csv(f'{PROJECT_DIR}/Falcon/postMN_final_filter/combined_library_matches.tsv', sep='\t')
classification = pd.read_csv(f'{PROJECT_DIR}/Falcon/postMN_final_filter/Classification_BAs_library_matches.tsv', sep='\t')

#merge
combined_df = combined_df.merge(classification[['SpectrumID', 'Is_BA_Checked']], on='SpectrumID', how='left')

In [ ]:
import re
from pathlib import Path

output_dir = Path(f'{PROJECT_DIR}/Falcon/postMN_final_filter/split_by_core')
output_dir.mkdir(parents=True, exist_ok=True)

required_cols = ['Core', '#Scan#', 'Is_BA_Checked']
missing_cols = [col for col in required_cols if col not in combined_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in combined_df: {missing_cols}")

export_df = combined_df[required_cols].copy()

for core_value, core_df in export_df.groupby('Core', dropna=False):
    core_name = 'NA' if pd.isna(core_value) else str(core_value)
    safe_core_name = re.sub(r'[^A-Za-z0-9._-]+', '_', core_name).strip('_') or 'unknown_core'
    out_file = output_dir / f"{safe_core_name}.tsv"
    core_df[['#Scan#', 'Is_BA_Checked']].to_csv(out_file, sep='\t', index=False)

print(f"Exported {export_df['Core'].nunique(dropna=False)} files to: {output_dir}")